# 成员B：属性识别可靠性诊断

只使用原150张固定样本。阶段1：人工盲标开发集；阶段2：运行开发集预测与评价；阶段3：复核开发集并冻结方案；阶段4：独立评估100张。Run All默认不会冻结方案或运行评估集。当前没有真实标签/预测指标，Notebook仅做过本地语法检查及离线逻辑测试，尚未在Kaggle GPU实测。

Kaggle设置：Internet开启；GPU T4。右侧Upload先添加本地交付包中的attribute_pilot_150_png.zip（只含150张原图，无权重）。人工标签准备好后再上传labels_dev_你的代号.csv；评估阶段再上传独立标注的labels_eval_代号.csv及保存的protocol_lock.json。


In [ ]:
from pathlib import Path
import subprocess,sys,json,csv,hashlib,datetime,zipfile,importlib.util
CODE=Path('/kaggle/working/memberb-attribute-reliability')
ASSETS=Path('/kaggle/working/team-baseline')
OUT=Path('/kaggle/working/attribute-reliability-'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
OUT.mkdir(parents=True,exist_ok=True)
if not CODE.exists():
    subprocess.run(['git','clone','--branch','memberb','https://github.com/Ariel-tian95/Composed-Image-Retrieval.git',str(CODE)],check=True)
SCRIPT=CODE/'scripts/attribute_pilot.py'
MANIFEST=CODE/'docs/attribute-pilot/labels.csv'
assert SCRIPT.exists() and 'def verify_lock' in SCRIPT.read_text(), '需要memberb最新版代码；请保留旧结果后使用新目录'
subprocess.run([sys.executable,'-m','pip','install','ftfy','regex','einops','einops-exts','Pillow','gdown'],check=True)
print('Code commit:',subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip())
print('本次输出目录:',OUT)


## 恢复原图和官方权重

先复用team-baseline。没有图片时从上传的小ZIP恢复；没有权重时先寻找已挂载official_checkpoint.pt，仍没有则下载官方2.53GB权重。不会重新下载FashionIQ全集，不重新跑检索。


In [ ]:
with MANIFEST.open(encoding='utf-8-sig') as f: manifest=list(csv.DictReader(f))
assert len(manifest)==150 and sum(r['split']=='dev' for r in manifest)==50
image_root=ASSETS/'data/fashion_iq/images'
image_root.mkdir(parents=True,exist_ok=True)
needed={r['image_id']+'.png':r['sha256'] for r in manifest}
missing=[n for n in needed if not (image_root/n).exists()]
if missing:
    archives=list(Path('/kaggle/input').rglob('attribute_pilot_150_png.zip'))
    assert len(archives)==1,'请上传交付目录中的attribute_pilot_150_png.zip，并确认右侧Input已挂载'
    with zipfile.ZipFile(archives[0]) as z:
        members={Path(i.filename).name:i for i in z.infolist() if not i.is_dir()}
        for name in missing:
            assert name in members
            content=z.read(members[name])
            assert hashlib.sha256(content).hexdigest()==needed[name], '原图哈希不一致: '+name
            (image_root/name).write_bytes(content)
for name,sha in needed.items():
    assert hashlib.sha256((image_root/name).read_bytes()).hexdigest()==sha,name
checkpoint=ASSETS/'checkpoints/context_i2w/official_checkpoint.pt'
if not checkpoint.exists():
    candidates=list(Path('/kaggle/input').rglob('official_checkpoint.pt'))
    assert len(candidates)<=1,'有多个权重，请显式选择一个并恢复到team-baseline/checkpoints/context_i2w/'
    if candidates:
        checkpoint.parent.mkdir(parents=True,exist_ok=True)
        checkpoint.symlink_to(candidates[0])
    else:
        subprocess.run([sys.executable,str(CODE/'scripts/download_context_checkpoint.py'),'--root',str(ASSETS)],check=True)
official=ASSETS/'external/context_i2w'
if not official.exists():
    official.parent.mkdir(parents=True,exist_ok=True)
    subprocess.run(['git','clone','https://github.com/Pter61/context-i2w.git',str(official)],check=True)
    subprocess.run(['git','-C',str(official),'checkout','c524d5b9f1551ef14e26791b2bb21654319d0493'],check=True)
assert subprocess.check_output(['git','-C',str(official),'rev-parse','HEAD'],text=True).strip()=='c524d5b9f1551ef14e26791b2bb21654319d0493'
import torch
assert torch.cuda.is_available(),'请先设置GPU，再开始预测'
print('原150张哈希全部通过；GPU:',torch.cuda.get_device_name(0))


## 开发集预测：只运行50张

请先在本地盲标开发集，不看模型答案。这里保留每个候选余弦分数、预测与top-2差值，差值不是校准概率。不会将预测填入人工标签。


In [ ]:
DEV_PRED=OUT/'dev_predictions.json'
subprocess.run([sys.executable,str(SCRIPT),'predict','--root',str(ASSETS),'--manifest',str(MANIFEST),'--split','dev','--out',str(DEV_PRED)],check=True)
print('已保存原始开发集预测:',DEV_PRED)


## 载入人工CSV，计算开发集指标

导出文件名以labels_dev_或labels_eval_开头。若发现同图不同意见，合并器生成冲突记录并停止，不自动挑选意见。这里只读取人工CSV，不搜索/使用模型输出作真值。无真值时写pending状态，不虚构准确率。


In [ ]:
LABEL_INPUTS=[]
for p in Path('/kaggle/input').rglob('labels_*.csv'):
    with p.open(encoding='utf-8-sig') as f: rs=list(csv.DictReader(f))
    if rs and 'reviewer' in rs[0] and any(r.get('reviewer','').strip() and any(r.get(g,'').strip() for g in ['color','sleeve_length','pattern']) for r in rs): LABEL_INPUTS.append(p)
LABEL_INPUTS=sorted(LABEL_INPUTS)
print('发现人工CSV:',*[str(p) for p in LABEL_INPUTS],sep='\n')
LABELS=OUT/'labels_merged.csv'
DEV_METRICS=OUT/'dev_metrics.json'
if LABEL_INPUTS:
    subprocess.run([sys.executable,str(CODE/'scripts/attribute_labels.py'),'--manifest',str(MANIFEST),'--inputs',*[str(p) for p in LABEL_INPUTS],'--out',str(LABELS)],check=True)
    subprocess.run([sys.executable,str(SCRIPT),'evaluate','--manifest',str(MANIFEST),'--labels',str(LABELS),'--predictions',str(DEV_PRED),'--split','dev','--out',str(DEV_METRICS)],check=True)
    print(DEV_METRICS.read_text())
else:
    (OUT/'evaluation_status.json').write_text(json.dumps({'status':'pending_labels','metrics':None,'next':'盲标并上传labels_dev_代号.csv'},ensure_ascii=False,indent=2))
    print('尚无人工真值：只保存预测，不计算准确率/F1。')


## 人工复核开发集后冻结方案

先查看开发集指标和错误案例。只用开发集调整提示词/规则；若改代码需重新跑开发集，再冻结。此版本没有选择拒识阈值，不声称预测可直接用于反馈。

确认后将FREEZE_PROTOCOL设为True并填入自己的代号。冻结前必须完成开发集50行的三属性和标注者；unknown/uncertain/not_visible允许，但不计入准确率。


In [ ]:
FREEZE_PROTOCOL=False
FREEZE_REVIEWER=''  # 由实际复核者填写，不代签
LOCK=OUT/'protocol_lock.json'
if FREEZE_PROTOCOL:
    assert DEV_METRICS.exists() and FREEZE_REVIEWER.strip(),'请先完成开发集评价并填写实际复核者'
    subprocess.run([sys.executable,str(SCRIPT),'freeze','--manifest',str(MANIFEST),'--labels',str(LABELS),'--reviewer',FREEZE_REVIEWER,'--out',str(LOCK)],check=True)
    print('务必下载保存:',LOCK)
else:
    print('未冻结方案，评估集不会运行。')


## 独立评估100张：默认不运行

评估集由另一成员独立盲标为佳。先导入其CSV并保留分歧处理记录；如果是新Kaggle会话，请上传保存的protocol_lock.json，并确保运行相同代码版本。不得根据评估集结果反复调参。


In [ ]:
RUN_EVAL=False
if RUN_EVAL:
    if not LOCK.exists():
        locks=list(Path('/kaggle/input').rglob('protocol_lock.json'))
        assert len(locks)==1,'请挂载开发集复核后保存的protocol_lock.json'
        LOCK=locks[0]
    assert LABELS.exists(),'请先上传人工CSV并运行合并单元格'
    with LABELS.open(encoding='utf-8-sig') as f: eval_rows=[r for r in csv.DictReader(f) if r['split']=='eval']
    assert len(eval_rows)==100 and all(r['reviewer'].strip() and all(r[g].strip() for g in ['color','sleeve_length','pattern']) for r in eval_rows),'请完成评估集100行人工标注'
    EVAL_PRED=OUT/'eval_predictions.json'
    subprocess.run([sys.executable,str(SCRIPT),'predict','--root',str(ASSETS),'--manifest',str(MANIFEST),'--split','eval','--lock',str(LOCK),'--out',str(EVAL_PRED)],check=True)
    subprocess.run([sys.executable,str(SCRIPT),'evaluate','--manifest',str(MANIFEST),'--labels',str(LABELS),'--predictions',str(EVAL_PRED),'--split','eval','--lock',str(LOCK),'--out',str(OUT/'eval_metrics.json')],check=True)
    print((OUT/'eval_metrics.json').read_text())
else:
    print('未运行评估集；目前不能宣称第⑤项完成。')


## 保存结果

输出目录含原始预测JSON、指标JSON/Markdown、错误案例CSV、合并人工标签以及方案锁。保留原始人工导出CSV和独立标注/分歧记录。保存Kaggle版本并下载ZIP，不能只依赖临时会话。


In [ ]:
import shutil
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('右侧Output中下载:',archive)
print('如下载链接404，先Save Version保存输出，等待完成，再到该保存版本的Output下载。')
